In [1]:
!pip install gensim nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 46.3 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import nltk
import gensim
import re

from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [3]:
sentences = [
    'This is the first document.',
    'This document is the second document.',
    'And this is the third one.',
    'Is this the first document?'
]

tfidf_vectorizer = TfidfVectorizer()
vector_matrix = tfidf_vectorizer.fit_transform(sentences)

cosine_similarity_matrix = cosine_similarity(vector_matrix)

df_cosine = pd.DataFrame(
    data=cosine_similarity_matrix,
    index=sentences,
    columns=sentences
)

df_cosine

,This is the first document.,This document is the second document.,And this is the third one.,Is this the first document?
This is the first document.,1.000000,0.646926,0.307772,1.000000
This document is the second document.,0.646926,1.000000,0.225240,0.646926
And this is the third one.,0.307772,0.225240,1.000000,0.307772
Is this the first document?,1.000000,0.646926,0.307772,1.000000


In [4]:
doc_1 = 'data science is one of the most important fields of science'
doc_2 = 'this is one of the best data science courses'
doc_3 = 'data scientists analyze data'

data = [doc_1, doc_2, doc_3]

tfidf = TfidfVectorizer()

result = tfidf.fit_transform(data)

df = pd.DataFrame(
    result.toarray(),
    columns=tfidf.get_feature_names_out()
)

df

,analyze,best,courses,data,fields,important,is,most,of,one,science,scientists,the,this
0,0.000000,0.000000,0.000000,0.189526,0.320895,0.320895,0.244049,0.320895,0.488098,0.244049,0.488098,0.000000,0.244049,0.000000
1,0.000000,0.400294,0.400294,0.236420,0.000000,0.000000,0.304434,0.000000,0.304434,0.304434,0.304434,0.000000,0.304434,0.400294
2,0.542701,0.000000,0.000000,0.641055,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.542701,0.000000,0.000000


In [5]:
df = pd.read_csv('/content/simpsons_script_lines.csv')

df.head()

,raw_character_text,spoken_words
0,Miss Hoover,"No, actually, it was a little of both. Sometim..."
1,Lisa Simpson,Where's Mr. Bergstrom?
2,Miss Hoover,I don't know. Although I'd sure like to talk t...
3,Lisa Simpson,That life is worth living.
4,Edna Krabappel-Flanders,The polls will be open from now until the end ...


In [6]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[0-9]", '', text)
    text = re.sub(r"[)(,”“.’$-]", '', text)
    return text

In [7]:
df = df.dropna(subset=['spoken_words'])

tokens = []

for i in df['spoken_words']:
    cleaned_text = clean_text(str(i))
    token = word_tokenize(cleaned_text)
    tokens.append(token)

In [8]:
Skip_gram_model = gensim.models.Word2Vec(
    tokens,
    min_count=1,
    vector_size=100,
    window=5,
    sg=1
)

In [9]:
print("Words similar to homer:")
print(Skip_gram_model.wv.most_similar('homer'))

Words similar to homer:
[('marge', 0.8621653914451599), ('abe', 0.8580641150474548), ('bartholomew', 0.8140198588371277), ('grampa', 0.8134809136390686), ('bart', 0.8074116110801697), ('eliza', 0.8047159910202026), ('mister', 0.803608775138855), ('jessica', 0.8017983436584473), ('jay', 0.7999259233474731), ('sweetheart', 0.798040509223938)]


In [10]:
print("Words similar to marge:")
print(Skip_gram_model.wv.most_similar('marge'))

Words similar to marge:
[('homer', 0.8621652126312256), ('abe', 0.8397844433784485), ('becky', 0.8250673413276672), ('sweetheart', 0.8232392072677612), ('sweetie', 0.8205733895301819), ('honey', 0.8069290518760681), ('maude', 0.8001199960708618), ('lisa', 0.7962659001350403), ('midge', 0.7934489250183105), ('bea', 0.7913233041763306)]


In [11]:
print(Skip_gram_model.wv.doesnt_match(
    ['jimbo', 'milhouse', 'kearney']
))

milhouse


In [12]:
print(Skip_gram_model.wv.doesnt_match(
    ['nelson', 'bart', 'milhouse']
))

nelson


In [13]:
print(Skip_gram_model.wv.doesnt_match(
    ['homer', 'patty', 'selma']
))

homer
